# 手寫簽名辨識評估

此 Notebook 用於評估多個 LLM 模型對於 `sample` 資料夾中手寫繁體中文簽名的辨識能力。
 
評估指標包括：
1. **單字正確率 (Character Accuracy)**
2. **姓名完全正確率 (Exact Match Accuracy)**
3. **模型穩定性 (Consistency/Stability)**: 同一張圖片測試3次，結果的一致性。

In [ ]:
import os
import base64
import time
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
import Levenshtein  # 用於計算單字正確率 (編輯距離)

load_dotenv()

## 1. 載入資料
讀取 `sample` 資料夾中的圖片，檔名即為正確答案（Ground Truth）。

In [ ]:
sample_dir = Path("sample")
if not sample_dir.exists():
    print("請確保 sample 資料夾存在")

# 讀取所有圖片檔案 (支援 png, jpg, jpeg)
image_files = []
for ext in ["*.png", "*.jpg", "*.jpeg"]:
    image_files.extend(list(sample_dir.glob(ext)))

dataset = []
for img_path in image_files:
    ground_truth = img_path.stem  # 檔名即為正確答案
    dataset.append({
        "image_path": img_path,
        "ground_truth": ground_truth
    })

print(f"共找到 {len(dataset)} 張圖片。")

## 2. 模型初始化與 API 封裝

In [ ]:
# --- Prompt 優化設定 ---
SYSTEM_INSTRUCTION = "你是一個專業的筆跡鑑定與繁體中文手寫辨識專家。你的唯一任務是精準還原圖片中的手寫簽名（通常為 2 到 4 個字的中文姓名）。"

USER_PROMPT = """請仔細分析圖中繁體中文手寫簽名的筆畫與結構，並辨識出正確的姓名。
嚴格輸出規則：
1. 【絕對限制】只能輸出純文字的姓名結果，例如「陳小明」。
2. 【禁止事項】絕不可包含任何解釋、問候語（如「圖片中是...」、「簽名為...」）、引號、標點或換行符號。
3. 【推測機制】若遇草書、連筆或難以辨識的字元，請結合筆畫輪廓與繁體中文常見姓名用字，給出最合理的推測，絕對不要填寫問號等無法辨識的符號。"""

# 1. Azure OpenAI (GPT-5.4)
from openai import AzureOpenAI

azure_openai_client = AzureOpenAI(
    api_version="2025-04-01-preview",
    azure_endpoint="https://project-04-openai-service.openai.azure.com/",
    api_key=os.getenv("5.4_AZURE_OPENAI_API_KEY"),
)

def run_azure_openai_gpt54(image_path):
    with open(image_path, "rb") as image_file:
        encoded_string = base64.b64encode(image_file.read()).decode("utf-8")
    
    response = azure_openai_client.responses.create(
        model="project-04-gpt-5.4",
        instructions=SYSTEM_INSTRUCTION,
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": USER_PROMPT},
                    {"type": "input_image", "image_url": f"data:image/png;base64,{encoded_string}", "detail": "high"}, 
                ],
            }
        ],
        max_output_tokens=50,
    )
    return response.output_text.strip()


# 2. Google Gemini
from google import genai
from google.genai import types

gemini_client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

def extract_gemini_text(response):
    candidates = getattr(response, "candidates", None) or []
    if not candidates:
        return ""
    parts = getattr(candidates[0].content, "parts", None) or []
    return "".join(part.text for part in parts if getattr(part, "text", None)).strip()

def run_gemini(model_name, image_path):
    image_bytes = image_path.read_bytes()
    image_part = types.Part.from_bytes(data=image_bytes, mime_type="image/png")
    
    combined_prompt = f"{SYSTEM_INSTRUCTION}\n\n{USER_PROMPT}"
    response = gemini_client.models.generate_content(
        model=model_name,
        contents=[combined_prompt, image_part],
        config=types.GenerateContentConfig(
            temperature=0.2,
            max_output_tokens=50,
        ),
    )
    return extract_gemini_text(response)


# 3. Anthropic Azure (Claude)
from anthropic import AnthropicFoundry

claude_opus_client = AnthropicFoundry(
    api_key=os.getenv("4.5_OPUS_AZURE_ANTHROPIC_API_KEY"),
    base_url="https://project3-docai-resource.services.ai.azure.com/anthropic/",
)

claude_sonnet_client = AnthropicFoundry(
    api_key=os.getenv("4.6_SONNET_AZURE_ANTHROPIC_API_KEY"),
    base_url="https://9h00200-act-aifoundry.openai.azure.com/anthropic/",
)

def run_anthropic(client, deployment_name, image_path):
    with open(image_path, "rb") as image_file:
        encoded_string = base64.b64encode(image_file.read()).decode("utf-8")
        
    message = client.messages.create(
        model=deployment_name,
        system=SYSTEM_INSTRUCTION,
        messages=[
            {
                "role": "user", 
                "content": [
                    {"type": "text", "text": USER_PROMPT},
                    {
                        "type": "image",
                        "source": {
                            "type": "base64",
                            "media_type": "image/png",
                            "data": encoded_string
                        }
                    }
                ]
            }
        ],
        max_tokens=50,
        temperature=0.2,
    )
    return message.content[0].text.strip()

## 3. 測試函式封裝與執行

In [ ]:
models = [
    {"name": "GPT-5.4", "func": lambda img: run_azure_openai_gpt54(img)},
    {"name": "gemini-3.1-flash-lite-preview", "func": lambda img: run_gemini("gemini-3.1-flash-lite-preview", img)},
    {"name": "gemini-3-flash-preview", "func": lambda img: run_gemini("gemini-3-flash-preview", img)},
    {"name": "gemini-3.1-pro-preview", "func": lambda img: run_gemini("gemini-3.1-pro-preview", img)},
    {"name": "claude-opus-4-5", "func": lambda img: run_anthropic(claude_opus_client, "claude-opus-4-5", img)},
    {"name": "claude-sonnet-4-6", "func": lambda img: run_anthropic(claude_sonnet_client, "project-04-claude-sonnet-4-6", img)},
]

NUM_RUNS = 3
results = []

for data in dataset:
    img_path = data["image_path"]
    gt = data["ground_truth"]
    print(f"正在處理圖片: {img_path.name} (正確答案: {gt})")
    
    for model_info in models:
        model_name = model_info["name"]
        func = model_info["func"]
        
        for run_idx in range(NUM_RUNS):
            try:
                pred = func(img_path)
            except Exception as e:
                print(f"[{model_name}] 處理失敗: {e}")
                pred = ""
                time.sleep(2)
                
            results.append({
                "image": img_path.name,
                "ground_truth": gt,
                "model": model_name,
                "run_idx": run_idx + 1,
                "prediction": pred
            })
            # 避免 API rate limit
            time.sleep(1)

## 4. 計算評估指標

利用 `Levenshtein` 計算單字正確率 (1 - 正規化編輯距離)。
利用精確比對計算姓名完全正確率。

In [ ]:
df_results = pd.DataFrame(results)

def calculate_char_accuracy(gt, pred):
    if not gt:
        return 0.0
    distance = Levenshtein.distance(gt, pred)
    max_len = max(len(gt), len(pred))
    if max_len == 0: return 1.0
    return max(0.0, 1.0 - (distance / max_len))

# 移除預測結果中的空白等可能干擾的字元
df_results['prediction_clean'] = df_results['prediction'].str.replace(r"\s+", "", regex=True)

df_results['exact_match'] = df_results['ground_truth'] == df_results['prediction_clean']
df_results['char_accuracy'] = df_results.apply(lambda row: calculate_char_accuracy(row['ground_truth'], row['prediction_clean']), axis=1)

display(df_results.head(10))

## 5. 統計與視覺化

In [ ]:
# 1. 姓名完全正確率 (Exact Match Accuracy) 依模型統計
exact_match_summary = df_results.groupby('model')['exact_match'].mean().reset_index()
exact_match_summary.rename(columns={'exact_match': 'Exact Match Accuracy'}, inplace=True)

# 2. 單字正確率 (Character Accuracy) 依模型統計
char_acc_summary = df_results.groupby('model')['char_accuracy'].mean().reset_index()
char_acc_summary.rename(columns={'char_accuracy': 'Character Accuracy'}, inplace=True)

# 3. 模型穩定性 (Consistency)
# 判斷同一張圖片的三次預測是否完全一致
def check_consistency(group):
    preds = group['prediction_clean'].tolist()
    return len(set(preds)) == 1

consistency_summary = df_results.groupby(['model', 'image']).apply(check_consistency).reset_index(name='is_consistent')
consistency_summary = consistency_summary.groupby('model')['is_consistent'].mean().reset_index()
consistency_summary.rename(columns={'is_consistent': 'Stability (Consistency)'}, inplace=True)

# 合併報表
final_report = pd.merge(exact_match_summary, char_acc_summary, on='model')
final_report = pd.merge(final_report, consistency_summary, on='model')

print("========== 模型評估總表 ==========")
display(final_report)

# 儲存結果
df_results.to_csv("evaluation_raw_results.csv", index=False, encoding="utf-8-sig")
final_report.to_csv("evaluation_summary.csv", index=False, encoding="utf-8-sig")
print("\n已儲存詳細結果至 evaluation_raw_results.csv，統計結果至 evaluation_summary.csv")